# urban-energy-core Capabilities Walkthrough

This notebook demonstrates the package as a standalone workflow.

For now, it points to a local external data root that contains the current project data.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from urban_energy_core.io import (
    load_all_fsa_census,
    load_city_fsa_geojsons,
    load_city_weather_csvs,
    load_processed_electricity_wide,
)
from urban_energy_core.pipelines import build_cities_from_data

## 2. Configure Data Root

By default this notebook uses the current external data root on disk.
You can change `DATA_ROOT` later without changing the package itself.

In [ ]:
DATA_ROOT = REPO_ROOT.parent / "urban-energy-data"
ELEC_PATH = DATA_ROOT / "data" / "processed" / "electricity" / "elec_rebuilt_new_weather.parquet"
CENSUS_ROOT = DATA_ROOT / "data" / "raw" / "census" / "FSA scale"
GEOMETRY_ROOT = DATA_ROOT / "data" / "raw" / "geometry"
WEATHER_ROOT = DATA_ROOT / "data" / "raw" / "weather"

for path in [DATA_ROOT, ELEC_PATH, CENSUS_ROOT, GEOMETRY_ROOT, WEATHER_ROOT]:
    print(path, "->", path.exists())

## 3. Load Inputs

In [ ]:
elec_df = load_processed_electricity_wide(ELEC_PATH)
census_df = load_all_fsa_census(root_dir=CENSUS_ROOT, drop_key_col=False, show_progress=False)
if "GEO UID" in census_df.columns:
    census_df = census_df.set_index("GEO UID")
census_df.index = census_df.index.astype(str)

geo = load_city_fsa_geojsons(geometry_dir=GEOMETRY_ROOT, show_progress=False)
weather = load_city_weather_csvs(weather_dir=WEATHER_ROOT, show_progress=False)

print("Electricity shape:", elec_df.shape)
print("Census shape:", census_df.shape)
print("Cities in geometry:", sorted(geo.keys()))
print("Cities in weather:", sorted(weather.keys()))

## 4. Build City Objects

In [ ]:
cities = build_cities_from_data(
    elec_df=elec_df,
    city_geojsons=geo,
    city_weather=weather,
    census_df=census_df,
    show_progress=False,
)

sorted(cities.keys())

## 5. Inspect Montreal

In [ ]:
montreal = cities["montreal"]
first_fsa_code = montreal.list_fsa_codes()[0]
first_fsa = montreal.get_fsa(first_fsa_code)

{
    "montreal_fsa_count": len(montreal.fsas),
    "example_fsa": first_fsa_code,
    "electricity_frame_shape": montreal.electricity_frame().shape,
}

## 6. Per-FSA Operations

In [ ]:
normalized = first_fsa.normalize_for_weather(montreal.weather)
per_capita = first_fsa.per_capita_consumption()
heating_prism = first_fsa.apply_heating_prism(montreal.weather)
short_term_daily = first_fsa.short_term_metrics(show_progress=False)

print("Normalized length:", len(normalized))
print("Per-capita mean:", float(per_capita.mean()))
print("Heating change point:", heating_prism["heating_change_point_temp_c"])
print("Short-term rows:", len(short_term_daily))

In [ ]:
short_term_daily.head()

## 7. City-Level Tables

In [ ]:
prism_table = montreal.compute_prism_table(show_progress=False)
short_term_table = montreal.compute_short_term_table(show_progress=False)

print("PRISM table shape:", prism_table.shape)
print("Short-term table shape:", short_term_table.shape)

In [ ]:
prism_table[["heating_slope_per_hdd", "r2"]].head(10)

In [ ]:
short_term_table.head(10)

## 8. Map Object

In [ ]:
map_fig = montreal.plot_map(metric="mean", alpha=0.4, figsize=(8, 5), show=False)
map_fig

## 9. Example Interpretation

In [ ]:
strongest = prism_table["heating_slope_per_hdd"].idxmax()
strongest_row = prism_table.loc[strongest]

print(
    f"Highest heating-sensitive Montreal FSA: {strongest} "
    f"(slope={strongest_row['heating_slope_per_hdd']:.4f}, r2={strongest_row['r2']:.3f})"
)

## 10. Notes

- This notebook presents `urban-energy-core` as a standalone package.
- The current `DATA_ROOT` is just the local source of data for now.
- Later, the same notebook can point at package-native data locations or a different external root.